In [0]:
%sql

create or replace table banking.gold.branch_performance as

with customer_branch as (
    select c.customer_id, c.branch_code from banking.silver.customers c
),

account_agg as (
    select a.customer_id, count(*) as total_accounts, sum(a.balance) as total_balance from banking.silver.accounts a
    group by a.customer_id
),

txn_agg as (
    select  A.customer_id, count(T.txn_id) as total_transactions,
    sum(T.amount) as total_transaction_amount
    from banking.silver.transactions T
    join banking.silver.accounts A   
    on T.account_id=A.account_id
    group by A.customer_id
)

 SELECT
     b.branch_code,
     b.branch_name,
     COUNT(DISTINCT cb.customer_id) AS total_customers,
     SUM(a.total_accounts) AS total_accounts,
     SUM(a.total_balance) AS total_deposits,
     SUM(t.total_transactions) AS total_transactions,
     SUM(t.total_transaction_amount) AS total_transaction_amount

 FROM banking.silver.branches b

 LEFT JOIN customer_branch cb
     ON b.branch_code = cb.branch_code

 LEFT JOIN account_agg a
     ON cb.customer_id = a.customer_id

 LEFT JOIN txn_agg t
     ON cb.customer_id = t.customer_id

 GROUP BY
 b.branch_code,
 b.branch_name

In [0]:
count=spark.sql("""
             select count(*) as cnt from banking.gold.branch_performance    
                """).collect()[0]['cnt']
dbutils.notebook.exit(str(count))